## Importando as bibliotecas necessárias

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import csv

print(np.__version__)

2.3.2


### Preparando o dataset

Nosso dataset está em um arquivo CSV chamado dados.csv, contendo 3 colunas:

- idade
- salário
- compra (rótulo: 1 para compra, 0 para não compra)

In [4]:
# função para carregar dados do csv
def carregar_dados(arquivo):
    x, y = [], []
    with open(arquivo, 'r') as f:
        leitor = csv.reader(f)
        next(leitor) # pular o cabeçalho
        for linha in leitor:
            x.append([float(linha[0]), float(linha[1])]) # idade, salário
            y.append(int(linha[2]))
    return np.array(x), np.array(y)

# carregando dataset
x, y = carregar_dados('dados.csv')
print("formato dos dados: ", x.shape, y.shape)
print("Primeiras amostras: \n", x[:5], "\nRótulos: ", y[:5])

formato dos dados:  (10, 2) (10,)
Primeiras amostras: 
 [[  22. 2000.]
 [  25. 2200.]
 [  47. 4300.]
 [  52. 4600.]
 [  46. 4100.]] 
Rótulos:  [0 0 1 1 1]


Obtendo os dados, mas dessa vez usando o loadtxt da numpy

In [6]:
dataset = np.loadtxt('dados.csv', delimiter=',', skiprows=1)

# separa as colunas de características (features), e a coluna de rótulos (labels)
# x_train recebe todas as linhas, mas apenas as duas primeiras colunas (indíce 0 e 1)
x_train = dataset[:, :2]

# y_train recebe todas as linhas, mas apenas a última coluna (indice 2)
# .astype(int) para garantir que as classes sejam tratadas como inteiros
y_train = dataset[:, 2].astype(int)

print("amostra das características (x_train):")
print(x_train[:5])
print('\nAmostra dos rótulos (y_train):')
print(y_train[:5])

amostra das características (x_train):
[[  22. 2000.]
 [  25. 2200.]
 [  47. 4300.]
 [  52. 4600.]
 [  46. 4100.]]

Amostra dos rótulos (y_train):
[0 0 1 1 1]


## Função entropia

In [8]:
def entropia(y):
    classes, contagem = np.unique(y, return_counts=True)
    probs = contagem / len(y)
    return -np.sum(probs * np.log2(probs + 1e-9))

# teste de entropia
print('Entropia [0,0,0,0] -> ', entropia([0,0,0,0]))
print('Entropia [0,1,0,1] -> ', entropia([0,1,0,1]))

Entropia [0,0,0,0] ->  -1.4426951595367387e-09
Entropia [0,1,0,1] ->  0.99999999711461


## Divisão de dados

In [9]:
def dividir(x, y, feature, limiar):
    esquerda_idx = x[:, feature] <= limiar
    direita_idx = x[:, feature] > limiar
    return x[esquerda_idx], y[esquerda_idx], x[direita_idx], y[direita_idx]

# testando divisão por idade por <= 40
xe, ye, xd, yd = dividir(x, y, feature=0, limiar=40)
print('esquerda: ', ye)
print('direita: ', yd)

esquerda:  [0 0 0 0 1]
direita:  [1 1 1 1 1]


## implementação da arvore

1. calcular entropia atual
2. testar todos os possíveis limiares para cada feature
3. escolher a divisão com maior ganho de informação
4. repetir recursivamente até atingir profundidade máxima ou um nó puro

In [12]:
class No:
    def __init__(self, feature=None, limiar=None, esquerda=None, direita=None, valor=None):
        self.feature = feature
        self.limiar = limiar
        self.sequerda = esquerda
        self.direita = direita
        self.valor = valor # classe se for folha

class ArvoreDecisao:
    def __init__(self, max_profundidade=3, min_amostras=2):
        self.max_profundidade = max_profundidade
        self.min_amostras = min_amostras
        self.raiz = None

    def ajustar(self, x, y):
        self.raiz = self._crescer(x, y, profundidade=0)

    def _crescer(self, x, y, profundidade):
        num_amostras, num_features = x.shape
        num_class = len(np.unique(y))